In [22]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm =ChatOpenAI(
    model="gpt-5.6-luna",
    reasoning_effort="none",
)

llm.invoke([HumanMessage("잘 지냈어?")])

AIMessage(content='응, 잘 지냈어! 😊 너는 잘 지냈어?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 11, 'total_tokens': 29, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EKmzeIsACMCpcmaLzdKdQ2TRKapBJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a07246-3342-7ba3-9eab-4757ad45a59a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 18, 'total_tokens': 29, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [23]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time

In [24]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [25]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage(
        "사용자의 질문에 답하기 위해 tools를 사용한다. "
        "timezone에는 반드시 유효한 IANA 타임존을 전달한다. "
        "대한민국의 모든 지역(부산 포함)은 'Asia/Seoul'을 사용한다."
    ),
    HumanMessage("부산은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

# (6) 생성된 llm 답변 출력
print(messages)

[SystemMessage(content="사용자의 질문에 답하기 위해 tools를 사용한다. timezone에는 반드시 유효한 IANA 타임존을 전달한다. 대한민국의 모든 지역(부산 포함)은 'Asia/Seoul'을 사용한다.", additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 248, 'total_tokens': 274, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EKmzfcKi95F1y0qTskwijShRSKH46', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a07246-36c1-7241-957b-850f6a3ed84e-0', tool_calls=[{'name': 'g

In [26]:

for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재시각 2026-09-06 00:53:20 


[SystemMessage(content="사용자의 질문에 답하기 위해 tools를 사용한다. timezone에는 반드시 유효한 IANA 타임존을 전달한다. 대한민국의 모든 지역(부산 포함)은 'Asia/Seoul'을 사용한다.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 248, 'total_tokens': 274, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EKmzfcKi95F1y0qTskwijShRSKH46', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a07246-36c1-7241-957b-850f6a3ed84e-0', tool_calls=[{'name': 

In [27]:

llm_with_tools.invoke(messages)

AIMessage(content='부산은 지금 **2026년 9월 6일 오전 12시 53분**입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 311, 'total_tokens': 338, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EKmzgLEvSDU7izHJAuBeiVJ32aYVb', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a07246-3a20-7f50-88c7-d35006abfe9d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 311, 'output_tokens': 27, 'total_tokens': 338, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [28]:

from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")

In [29]:
import yfinance as yf

@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [30]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 402, 'total_tokens': 432, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EKmzhEZbqQy1W1IAdDUGOvbvlGagF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a07246-3e3c-7cc1-bc27-2eaf912227d7-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}, 'id': 'call_zJFcS9osk5WYNmpZgrFMpTip', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 402, 'output_tokens': 30, 'total_tokens': 432, 'input_token_det

In [31]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)

{'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}
content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2026-08-05 00:00:00-04:00 | 323.42 | 327.14 | 320.28 |  321.55 | 2.78208e+07 |           0 |              0 |\n| 2026-08-06 00:00:00-04:00 | 317.05 | 323    | 315.52 |  319.53 | 2.60338e+07 |           0 |              0 |\n| 2026-08-07 00:00:00-04:00 | 322.34 | 333.73 | 321.25 |  328.58 | 3.94923e+07 |           0 |              0 |\n| 2026-08-10 00:00:00-04:00 | 326.6  | 332.05 | 326.15 |  330.88 | 2.50038e+07 |           0 |              0 |\n| 2026-08-11 00:00:00-04:00 | 332.8  | 336.2  | 329.53 |  332.81 | 2.33452e+07 |           0 |              0 |\n| 2026-08-12 00:00:00-04:00 | 335    | 335.5  | 323.64 |  327.51 | 2.86989e+07 |           0 |              0 |\n| 2026-08-13 00:00:00-04:0

In [32]:
llm_with_tools.invoke(messages)

AIMessage(content='테슬라(TSLA)는 **한 달 전보다 올랐습니다.**\n\n- 8월 5일 종가: **321.55달러**\n- 9월 4일 종가: **354.08달러**\n- 변동: **약 32.53달러 상승**\n- 수익률: **약 +10.1%**\n\n※ 미국 시장 종가 기준이며, 환율 변동은 반영하지 않았습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 1830, 'total_tokens': 1931, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 1827, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EKmziS0WEXOpImwP8tKqqFq1xIgZ0', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a07246-4229-7fb0-9154-ff1135b99409-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1830, 'output_tokens': 101, 'total_tokens'

In [33]:
for c in llm.stream([HumanMessage("잘 지냈어? 한국 사회의 문제점에 대해 이야기해줘.")]):
    print(c.content, end='|') 

|잘| 지|냈|어|!| 한국| 사회|의| 문제|점|은| 여러| 가지|가| 서로| 얽|혀| 있다고| 볼| 수| 있어|.| 대표|적으로|는| 다음|과| 같|아|.

|1|.| **|저|출|생|과| 인|구| 감소|**|  
|  | 높은| 주|거|비|와| 사|교육|비|,| 불|안|정|한| 일|자리|,| 경|력| 단|절|,| 돌|봄| 부담| 때문에| 결|혼|과| 출|산|을| 미|루|거나| 포|기|하는| 사람이| 많|아|지고| 있어|.| 이는| 노동|력| 감소|와| 지역| 소|멸|,| 연|금|·|복|지| 부담| 증가|로| 이어|질| 수| 있어|.

|2|.| **|주|거| 불|안|과| 자|산| 격|차|**|  
|  | 집|값|과| 전|월|세| 비용|이| 소|득|에| 비|해| 높|고|,| 주|택| 보|유| 여부|에| 따라| 자|산| 격|차|가| 크게| 벌|어|져| 있어|.| 특히| 청|년|층|은| 독|립|과| 결|혼|,| 출|산|을| 계획|하기| 어려|운| 경우|가| 많|아|.

|3|.| **|교육| 경쟁|과| 사|교육| 의|존|**|  
|  | 대학| 입|시|와| 학|벌|이| 취|업|·|사회|적| 기|회|에| 큰| 영향을| 미|친|다는| 인|식|이| 강|해| 사|교육| 경쟁|이| 심|해|.| 가|정|의| 경제|력이| 교육| 기|회|로| 이어|지|면서| 계|층| 이동|이| 어렵|다는| 불|평|도| 커|지고| 있어|.

|4|.| **|노|동|시장| 이|중|구|조|**|  
|  | 대|기업|·|공|공|부|문| 정|규|직|과| 중|소|기업|·|비|정|규|직| 사이|의| 임|금|,| 복|지|,| 고|용| 안정|성| 차|이가| 크|다|.| 청|년| 취|업|난|과| 중|소|기업| 인|력| 부족|이| 동시에| 나타|나는| 것도| 이런| 구조|와| 관련|이| 있어|.

|5|.| **|고|령|화|와| 노|인| 빈|곤|**|  
|  | 한국|의| 노|인| 빈|곤|율|은| 높은| 편|이며|,| 은|퇴| 후|에도| 생|계를| 위해| 일|해야| 하는| 사람이| 많|아|.| 동시에|

In [34]:
messages = [
    SystemMessage(
        "사용자의 질문에 답하기 위해 tools를 사용한다. "
        "timezone에는 반드시 유효한 IANA 타임존을 전달한다. "
        "대한민국의 모든 지역(부산 포함)은 'Asia/Seoul'을 사용한다."
    ),
    HumanMessage("부산은 지금 몇시야?"),
]

response = llm_with_tools.stream(messages)

# 파편화된 tool_call 청크를 하나로 합치기 
is_first = True
for chunk in response:    
    print("chunk type: ", type(chunk))
    
    if is_first:
        is_first = False
        gathered = chunk
    else:
        gathered += chunk
    
    print("content: ", gathered.content, "tool_call_chunk", gathered.tool_calls)

messages.append(gathered)

chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_5PmPWxCU54xQWzMMLhtbVKMT', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_5PmPWxCU54xQWzMMLhtbVKMT', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_5PmPWxCU54xQWzMMLhtbVKMT', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': ''}, 'id': 'call_5PmPWxCU54xQWzMMLhtbVKMT', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': 'Asia'}, 'id': 'call_5PmPWxCU54xQWzMMLhtbVKMT', 'type': 'tool_c

In [35]:
gathered

AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai', 'finish_reason': 'tool_calls', 'model_name': 'gpt-5.6-luna', 'service_tier': 'default'}, id='lc_run--01a07246-62b1-7a23-98bb-14fbcbac51a0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_5PmPWxCU54xQWzMMLhtbVKMT', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 316, 'output_tokens': 26, 'total_tokens': 342, 'input_token_details': {'audio': 0, 'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"부산"}', 'id': 'call_5PmPWxCU54xQWzMMLhtbVKMT', 'index': 0, 'type': 'tool_call_chunk'}], chunk_position='last')

In [36]:

for tool_call in gathered.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # tool_dict를 사용하여 도구 이름으로 도구 함수를 선택
    print(tool_call["args"]) # 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재시각 2026-09-06 00:53:31 


[SystemMessage(content="사용자의 질문에 답하기 위해 tools를 사용한다. timezone에는 반드시 유효한 IANA 타임존을 전달한다. 대한민국의 모든 지역(부산 포함)은 'Asia/Seoul'을 사용한다.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai', 'finish_reason': 'tool_calls', 'model_name': 'gpt-5.6-luna', 'service_tier': 'default'}, id='lc_run--01a07246-62b1-7a23-98bb-14fbcbac51a0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_5PmPWxCU54xQWzMMLhtbVKMT', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 316, 'output_tokens': 26, 'total_tokens': 342, 'input_token_details': {'audio': 0, 'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"부산"}', 'id': 'call_5PmPW

In [37]:

for c in llm_with_tools.stream(messages):
    print(c.content, end='|')

|부|산|은| 지금| **|202|6|년| |9|월| |6|일| 오전| |12|시| |53|분|**|입니다|.||||